# Figure 5 – SOH estimation and the role of SOC (Dataset 2)

Data are read from `data/` (written by `run_all.py`). Every figure is saved as `Figure5_<nn>_<title>.png` when it is shown.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("data")
PREFIX = "Figure5"
_plt_show = plt.show
_n = [0]


def _title(fig):
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:50]
    return "figure"


def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _n[0] += 1
        name = f"{PREFIX}_{_n[0]:02d}_{_title(fig)}.png"
        fig.savefig(name, dpi=fig.dpi, bbox_inches="tight")
        print("saved:", name)
    _plt_show(*args, **kwargs)


plt.show = _save_then_show

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

d = pd.read_csv(DATA / "soh_predictions.csv")
df_test = d[["Mat", "No.", "SOC", "SOH"]]
y_true = list(df_test["SOH"].values)
y_oracle = list(d["y_oracle"].values)
y_no_soc = d["y_no_soc"].values

## Predicted vs. true SOH for each material (SOC-aware model)

In [ ]:
def plot_soh_parity_by_material(df_test, y_pred):

    y_pred = np.array(y_pred)

    for mat in df_test["Mat"].unique():

        idx = (df_test["Mat"].values == mat)
        y_t = df_test.loc[idx, "SOH"].values
        y_p = y_pred[idx]

        mask = ~np.isnan(y_p)
        if mask.sum() < 20:
            continue

        plt.figure(figsize=(3,2.5),dpi=600)
        plt.scatter(
            y_t[mask],
            y_p[mask],
            s=15,
            alpha=0.4,
            color='#8BCFB5'
        )

        lims = [
            min(y_t[mask].min(), y_p[mask].min()),
            max(y_t[mask].max(), y_p[mask].max())
        ]
        lims = [
            0.5,
            1.0
        ]
        plt.plot(lims, lims, "k--", lw=1)

        plt.xlabel("True SOH")
        plt.ylabel("Predicted SOH")
        plt.title(f"{mat}: SOH Prediction")
        plt.ylim(0.5,1.0)
        plt.xlim(0.5,1.0)

        plt.tight_layout()
        plt.show()

plot_soh_parity_by_material(df_test, y_oracle)

## SOH error distribution for each material (SOC-aware model)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_soh_error_by_material(df_test, y_pred):

    sns.set_style("white")

    y_pred = np.array(y_pred)

    for mat in df_test["Mat"].unique():

        idx = (df_test["Mat"].values == mat)
        err = y_pred[idx] - df_test.loc[idx, "SOH"].values
        err = err[~np.isnan(err)]

        if len(err) < 20:
            continue

        fig, ax = plt.subplots(
            figsize=(1.5, 1.3),
            dpi=600
        )

        fig.patch.set_alpha(0)
        ax.set_facecolor("none")
        ax.patch.set_alpha(0)

        sns.histplot(
            err,
            bins=30,
            kde=True,
            color="#B9DA9A",
            ax=ax
        )

        ax.axvline(0, color="k", linestyle="--", lw=0.5)

        ax.set_xlabel("SOH Error", fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        ax.set_title(f"{mat}: SOH Error", fontsize=10)

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.tick_params(axis='both', labelsize=8)

        plt.tight_layout()
        plt.show()


plot_soh_error_by_material(df_test, y_oracle)

## SOC-unaware model: predicted vs. true SOH and error distribution

In [ ]:
plot_soh_parity_by_material(df_test, y_no_soc)
plot_soh_error_by_material(df_test, y_no_soc)

## Benefit of SOC-aware SOH estimation (overall metrics, per-material metrics and relative R² improvement)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def compare_global_metrics(y_true, y_no_soc, y_soc):
    y_true = np.array(y_true)
    y_no_soc = np.array(y_no_soc)
    y_soc = np.array(y_soc)

    mask_no = ~np.isnan(y_no_soc)
    mask_soc = ~np.isnan(y_soc)

    print("=========== Overall SOH Performance ===========")
    print(f"SOC-unaware | MAE = {mean_absolute_error(y_true[mask_no], y_no_soc[mask_no]):.4f} "
          f"| R2 = {r2_score(y_true[mask_no], y_no_soc[mask_no]):.4f}")
    print(f"SOC-aware   | MAE = {mean_absolute_error(y_true[mask_soc], y_soc[mask_soc]):.4f} "
          f"| R2 = {r2_score(y_true[mask_soc], y_soc[mask_soc]):.4f}")

    print("\nImprovement:")
    print(f"↓ MAE improvement = "
          f"{mean_absolute_error(y_true[mask_no], y_no_soc[mask_no]) - mean_absolute_error(y_true[mask_soc], y_soc[mask_soc]):.4f}")
    print(f"↑ R2 improvement  = "
          f"{r2_score(y_true[mask_soc], y_soc[mask_soc]) - r2_score(y_true[mask_no], y_no_soc[mask_no]):.4f}")

In [ ]:
compare_global_metrics(y_true, y_no_soc, y_oracle)

import pandas as pd
import matplotlib.pyplot as plt

def compare_metrics_by_material(df_test, y_no_soc, y_soc):

    records = []

    for mat in df_test["Mat"].unique():
        idx = df_test["Mat"] == mat

        y_t = df_test.loc[idx, "SOH"].values
        y_n = np.array(y_no_soc)[idx]
        y_s = np.array(y_soc)[idx]

        mask_n = ~np.isnan(y_n)
        mask_s = ~np.isnan(y_s)

        if mask_n.sum() < 20 or mask_s.sum() < 20:
            continue

        records.append({
            "Mat": mat,
            "MAE_noSOC": mean_absolute_error(y_t[mask_n], y_n[mask_n]),
            "MAE_SOC": mean_absolute_error(y_t[mask_s], y_s[mask_s]),
            "R2_noSOC": r2_score(y_t[mask_n], y_n[mask_n]),
            "R2_SOC": r2_score(y_t[mask_s], y_s[mask_s]),
        })

    df_cmp = pd.DataFrame(records)
    df_cmp["MAE_improve"] = df_cmp["MAE_noSOC"] - df_cmp["MAE_SOC"]
    df_cmp["R2_improve"] = df_cmp["R2_SOC"] - df_cmp["R2_noSOC"]

    df_cmp["MAE_improve"] = (df_cmp["MAE_noSOC"] - df_cmp["MAE_SOC"])/df_cmp["MAE_noSOC"]
    df_cmp["R2_improve"] = (df_cmp["R2_SOC"] - df_cmp["R2_noSOC"])/df_cmp["R2_noSOC"]

    return df_cmp

df_cmp = compare_metrics_by_material(df_test, y_no_soc, y_oracle)
df_cmp

plt.figure(figsize=(3,2), dpi=600)
plt.bar(df_cmp["Mat"], df_cmp["R2_improve"], color="#D097BA")
plt.ylabel("MAE Reduction")
plt.title("Benefit of SOC-aware SOH Estimation")
plt.ylabel("R2 Improvement")
plt.title("Benefit of SOC-aware SOH Estimation")
plt.tight_layout()
plt.show()

## Absolute SOH error vs. SOC range

In [ ]:
def plot_soh_error_box_strip_vs_soc(
    df_test, y_pred,
    soc_bins=[0, 20, 40,60,80,100]
):

    df = df_test.copy()
    df["SOH_pred"] = y_pred
    df["abs_err"] = np.abs(df["SOH_pred"] - df["SOH"])

    df["SOC_bin"] = pd.cut(
        df["SOC"],
        bins=soc_bins,
        include_lowest=True
    )

    df = df.dropna(subset=["abs_err", "SOC_bin"])

    plt.figure(figsize=(3,2.4),dpi=600)

    sns.boxplot(
        data=df,
        x="SOC_bin",
        y="abs_err",
        showfliers=False,
        color="lightgray"
    )

    sns.stripplot(
        data=df,
        x="SOC_bin",
        y="abs_err",
        size=2,
        alpha=0.35,
        jitter=True,
        color="#80BEB3"
    )

    plt.xlabel("SOC Range (%)")
    plt.ylabel("|SOH Error|")
    plt.title("SOH Error vs SOC")

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

plot_soh_error_box_strip_vs_soc(df_test, y_oracle)

## Absolute SOH error for each material

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_soh_error_box_strip_by_material(df_test, y_pred):

    df = df_test.copy()
    df["SOH_pred"] = y_pred
    df["abs_err"] = np.abs(df["SOH_pred"] - df["SOH"])
    df = df.dropna(subset=["abs_err"])

    plt.figure(figsize=(3.6, 2.6), dpi=600)
    sns.boxplot(
        data=df,
        x="Mat",
        y="abs_err",
        showfliers=False,
        color="lightgray",
        width=0.55
    )

    sns.stripplot(
        data=df,
        x="Mat",
        y="abs_err",
        size=2.5,
        jitter=0.25,
        alpha=0.45,
        color="#80BEB3"
    )

    plt.xlabel("Material")
    plt.ylabel("|SOH Error|")
    plt.title("SOH Estimation Error by Material")

    plt.tight_layout()
    plt.show()

plot_soh_error_box_strip_by_material(df_test, y_oracle)

## Absolute SOH error for each material (second rendering)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_soh_error_box_strip_by_material(df_test, y_pred):

    df = df_test.copy()
    df["SOH_pred"] = y_pred
    df["abs_err"] = np.abs(df["SOH_pred"] - df["SOH"])
    df = df.dropna(subset=["abs_err"])

    plt.figure(figsize=(3.6, 2.6), dpi=600)

    sns.boxplot(
        data=df,
        x="Mat",
        y="abs_err",
        showfliers=False,
        color="lightgray",
        width=0.55
    )

    sns.stripplot(
        data=df,
        x="Mat",
        y="abs_err",
        size=2.5,
        jitter=0.25,
        alpha=0.45,
        color="#80BEB3"
    )

    plt.xlabel("Material")
    plt.ylabel("|SOH Error|")
    plt.title("SOH Estimation Error by Material")

    plt.tight_layout()
    plt.show()

plot_soh_error_box_strip_by_material(df_test, y_oracle)

## Feature-space data density

`data/df_res.csv`: for each number of pulse blocks *k* and battery-level training ratio, the mean kNN density of the test samples with respect to the training samples and the SOH MAE / R² (true material and true SOC).

In [ ]:
df_res = pd.read_csv(DATA / "df_res.csv")
df_res

## SOH MAE vs. data density (Figure 5j)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(6, 5),dpi=600)

feature_sets = sorted(df_res["n_blocks"].unique())
palette = sns.color_palette("viridis", n_colors=len(feature_sets))
color_map = dict(zip(feature_sets, palette))

size_min, size_max = 40, 180
r_min = df_res["train_ratio"].min()
r_max = df_res["train_ratio"].max()

def ratio_to_size(r):
    return size_min + (r - r_min) / (r_max - r_min) * (size_max - size_min)

for fs in feature_sets:
    sub = (
        df_res[df_res["n_blocks"] == fs]
        .sort_values("mean_density")
    )

    ax.plot(
        sub["mean_density"],
        sub["soh_mae"],
        color=color_map[fs],
        linewidth=2.0,
        alpha=0.85
    )

    ax.scatter(
        sub["mean_density"],
        sub["soh_mae"],
        s=sub["train_ratio"].apply(ratio_to_size),
        color=color_map[fs],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        label=fs
    )

    ax.axhline(
        y=0.03,
        color="gray",
        linestyle=":",
        linewidth=2.5
    )

ax.set_xlabel("Mean data density", fontsize=13)
ax.set_ylabel("SOH MAE", fontsize=13)
ax.set_title(
    "Density-driven SOH sufficiency",
    fontsize=14
)

handles_color = [
    plt.Line2D([0], [0], color=color_map[f], lw=3, label=f)
    for f in feature_sets
]
legend1 = ax.legend(
    handles=handles_color,
    title="Pulse Count",
    loc="upper right",
    frameon=False
)

for r in sorted(df_res["train_ratio"].unique()):
    ax.scatter(
        [], [],
        s=ratio_to_size(r),
        color="gray",
        edgecolor="black",
        label=f"Train ratio = {r}"
    )

ax.legend(
    title="Training data ratio",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=4,
    frameon=False
)
ax.set_ylim(0.023,0.055)
sns.despine()
plt.tight_layout()
plt.show()

## Marginal change in SOH error with increasing data density (Figure 5k)

In [ ]:
df_soh_delta = []

for nb in sorted(df_res["n_blocks"].unique()):
    sub = (
        df_res[df_res["n_blocks"] == nb]
        .sort_values("mean_density")
        .copy()
    )

    sub["delta_soh_mae"] = sub["soh_mae"].diff()
    df_soh_delta.append(abs(sub))

df_soh_delta = pd.concat(df_soh_delta, ignore_index=True)

fig, ax = plt.subplots(figsize=(5, 3.5), dpi=600)

for fs in feature_sets:
    sub = (
        df_soh_delta[df_soh_delta["n_blocks"] == fs]
        .dropna(subset=["delta_soh_mae"])
        .sort_values("mean_density")
    )

    ax.plot(
        sub["mean_density"],
        abs(-sub["delta_soh_mae"]),
        color=color_map[fs],
        linewidth=2.0,
        alpha=0.85
    )

    ax.scatter(
        sub["mean_density"],
        abs(-sub["delta_soh_mae"]),
        s=sub["train_ratio"].apply(ratio_to_size),
        color=color_map[fs],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9
    )

ax.axhline(
    y=0.002,
    color="gray",
    linestyle=":",
    linewidth=2.5
)

ax.set_xlabel("Mean data density", fontsize=13)
ax.set_ylabel("Marginal SOH error change", fontsize=13)
ax.set_title(
    "Diminishing returns of SOH accuracy with increasing data density",
    fontsize=14
)

sns.despine()
plt.tight_layout()
plt.show()